In [5]:
import mne
import numpy as np
import pandas as pd

In [2]:
# Carrega as épocas filtradas (1 - 30 Hz) geradas no Notebook 2
epochs = mne.read_epochs('epochs_vp001_filtradas-epo.fif', preload=True)

# Mantém apenas os 28 canais de EEG (remove HEOG e VEOG para não contaminar as features)
epochs.pick('eeg')

X = epochs.get_data()          # Shape: (540 épocas, 28 canais, 446 amostras)
y = epochs.events[:, 2]        # Rótulos: 0 = non_target, 1 = target
ch_names = epochs.ch_names     # Nomes dos 28 eletrodos
sfreq = epochs.info['sfreq']   # 200.0 Hz

# Transforma para o domínio da frequência (FFT)
# Usamos apenas a metade positiva do espectro (até Nyquist = 100 Hz), conforme a fórmula f_j = j * fs / (2 * M)
X_freq_full = np.fft.fft(X, axis=-1)
M = X.shape[-1] // 2
X_freq = X_freq_full[:, :, :M]

print("Shape no domínio do tempo:", X.shape)
print("Shape no domínio da frequência:", X_freq.shape)
print("Distribuição das classes -> Non-Target (0):", np.sum(y == 0), "| Target (1):", np.sum(y == 1))

Reading d:\Faculdade\Hubner\topicos-especiais\epochs_vp001_filtradas-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    2225.00 ms
        0 CTF compensation matrices available
Not setting metadata
540 matching events found
No baseline correction applied
0 projection items activated
Shape no domínio do tempo: (540, 28, 446)
Shape no domínio da frequência: (540, 28, 223)
Distribuição das classes -> Non-Target (0): 252 | Target (1): 288


In [3]:
# --- Funções Auxiliares ---
def PSD(w):
    """Densidade Espectral de Potência (Power Spectral Density)."""
    return np.abs(w) ** 2

def get_freqs(M, sfreq):
    """Calcula o vetor de frequências f_j = j * SampleRate / (2 * M)."""
    j = np.arange(1, M + 1)
    return (j * sfreq) / (2.0 * M)


# --- Características no Domínio do Tempo (Phinyomark et al., 2008) ---
def wamp(x, threshold=5e-6):
    """Willison Amplitude (WAMP): conta saltos consecutivos acima do limiar (padrão: 5 µV)."""
    diff = np.abs(np.diff(x, axis=-1))
    return np.sum(diff > threshold, axis=-1)

def var(x):
    """Variance (VAR): potência média do sinal na época."""
    N = x.shape[-1]
    return np.sum(x ** 2, axis=-1) / (N - 1)

def rms(x):
    """Root Mean Square (RMS): amplitude efetiva do sinal."""
    N = x.shape[-1]
    return np.sqrt(np.sum(np.abs(x) ** 2, axis=-1) / N)

def wl(x):
    """Waveform Length (WL): comprimento acumulado da onda no tempo."""
    return np.sum(np.abs(np.diff(x, axis=-1)), axis=-1)

def zc(x):
    """Zero Crossing (ZC): número de vezes que o sinal cruza o eixo zero."""
    sign_change = (x[:, :, :-1] * x[:, :, 1:]) <= 0
    return np.sum(sign_change, axis=-1)

def mav(x):
    """Mean Absolute Value (MAV) [Extra - Phinyomark et al., 2008]: média dos valores absolutos da amplitude."""
    return np.mean(np.abs(x), axis=-1)


# --- Características no Domínio da Frequência (Phinyomark et al., 2008) ---
def fmd(w):
    """Median Frequency (FMD): metade da soma total da potência espectral."""
    return np.sum(PSD(w), axis=-1) / 2.0

def fmn(w, sfreq):
    """Mean Frequency (FMN): frequência média ponderada pela potência (PSD)."""
    psd_vals = PSD(w)
    freqs = get_freqs(w.shape[-1], sfreq)
    return np.sum(psd_vals * freqs, axis=-1) / np.sum(psd_vals, axis=-1)

def mmdf(w):
    """Modified Median Frequency (MMDF): metade da soma total das amplitudes espectrais."""
    return np.sum(np.abs(w), axis=-1) / 2.0

def mmnf(w, sfreq):
    """Modified Mean Frequency (MMNF): frequência média ponderada pela amplitude linear."""
    amp_vals = np.abs(w)
    freqs = get_freqs(w.shape[-1], sfreq)
    return np.sum(amp_vals * freqs, axis=-1) / np.sum(amp_vals, axis=-1)

In [6]:
# Dicionário com todas as características extraídas
features_dict = {
    'WAMP': wamp(X, threshold=5e-6),
    'VAR':  var(X),
    'RMS':  rms(X),
    'WL':   wl(X),
    'ZC':   zc(X),
    'MAV':  mav(X),
    'FMD':  fmd(X_freq),
    'FMN':  fmn(X_freq, sfreq),
    'MMDF': mmdf(X_freq),
    'MMNF': mmnf(X_freq, sfreq),
}

nomes_features = list(features_dict.keys())
final = np.array(list(features_dict.values()))  # Shape: (10 features, 540 épocas, 28 canais)

# Reordena para (540 épocas, 28 canais, 10 features) e achata para 2D: (540, 280)
X_ord = final.transpose(1, 2, 0)
X_final = X_ord.reshape(X_ord.shape[0], X_ord.shape[1] * X_ord.shape[2])

print("Shape para classificação 3D:", X_ord.shape)
print("Shape da matriz tabular 2D:", X_final.shape)

# Cria os nomes das colunas organizados por canal (ex: WAMP_Fp1, VAR_Fp1, ..., MMNF_O2)
colunas = [f"{feat}_{ch}" for ch in ch_names for feat in nomes_features]

df_features = pd.DataFrame(X_final, columns=colunas)
df_features.insert(0, 'ID_sample', np.arange(1, len(df_features) + 1))
df_features['Classe'] = y

# Salva em CSV e NumPy para usar no próximo notebook de Machine Learning
df_features.to_csv('features_vp001_nback.csv', index=False)
np.save('X_features_nback.npy', X_final)
np.save('y_labels_nback.npy', y)

df_features.head()

Shape para classificação 3D: (540, 28, 10)
Shape da matriz tabular 2D: (540, 280)


,ID_sample,WAMP_Fp1,VAR_Fp1,RMS_Fp1,WL_Fp1,ZC_Fp1,MAV_Fp1,FMD_Fp1,FMN_Fp1,MMDF_Fp1,...,VAR_O2,RMS_O2,WL_O2,ZC_O2,MAV_O2,FMD_O2,FMN_O2,MMDF_O2,MMNF_O2,Classe
0,1,445.0,1790.386207,42.265493,1547.566389,35.0,23.540879,8.885245e+07,3.331589,29438.615034,...,36.150071,6.005749,726.635450,46.0,4.583993,1.798297e+06,8.063405,6440.550136,13.169692,0
1,2,445.0,2922.534545,53.999831,1568.386611,40.0,30.824698,1.453960e+08,2.835139,34530.242887,...,53.501221,7.306248,740.308484,38.0,5.737555,2.686434e+06,6.471732,7617.702643,12.045238,0
2,3,445.0,5355.538232,73.099455,2505.506932,15.0,54.811243,2.660806e+08,3.316717,46752.985522,...,41.131199,6.406167,790.212245,45.0,5.139921,2.041194e+06,8.883132,7174.992260,14.578323,0
3,4,445.0,2619.911035,51.127652,2015.033318,15.0,34.248828,1.300051e+08,3.759652,34361.145795,...,44.806910,6.686288,729.034992,45.0,5.107058,2.238179e+06,7.854813,7522.712903,13.813804,1
4,5,445.0,2048.715639,45.211969,1530.089097,26.0,26.083916,1.018739e+08,3.083952,31057.367365,...,44.722957,6.680021,673.123901,44.0,5.074972,2.225653e+06,7.213821,6975.924038,12.331424,0
